## Imports

In [ ]:
pip install -q transformers torch accelerate langchain langchain-core langchain-community pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 38.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Note: you may need

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json

## Model Loading

In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Text Generation

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,  
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Define Structured Output Schema

In [ ]:
from pydantic import BaseModel
from typing import Optional, Literal

class AgentDecision(BaseModel):
    intent: Literal[
        "calculator",
        "text_processing",
        "data_transformation"
    ]

    expression: Optional[str] = None
    text: Optional[str] = None

    operation: Optional[Literal[
        "km_to_miles",
        "c_to_f",
        "format_date",
        "validate_email"
    ]] = None

    value: Optional[str] = None

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=AgentDecision)

## Create Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a strict AI agent.

Your job:
1. Identify user intent.
2. Extract structured parameters.

Allowed intents:
- calculator
- text_processing
- data_transformation

Allowed operations for data_transformation:
- km_to_miles
- c_to_f
- format_date
- validate_email

Rules:
- Respond ONLY with valid JSON.
- No explanations.
- No extra text.
- Extract numeric values only (no units).

{format_instructions}

User input:
{user_input}
""",
    input_variables=["user_input"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [ ]:
# Build Decision Chain
decision_chain = prompt | llm

In [ ]:
def get_agent_decision(user_input: str):
    raw_output = decision_chain.invoke({"user_input": user_input})
    decision = parser.parse(raw_output)
    return decision

In [ ]:
decision = get_agent_decision("Convert 10 km to miles")
print(decision)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


intent='data_transformation' expression=None text=None operation='km_to_miles' value='10'


## Implement Tools

### Calculator Tool

In [ ]:
import math

def calculator_tool(expression: str):
    try:
        result = eval(expression, {"__builtins__": None}, {"math": math})
        return {"result": result}
    
    except ZeroDivisionError:
        return {"error": "Division by zero"}
    
    except Exception as e:
        return {"error": str(e)}

### Text Processing Tool

In [ ]:
def text_processing_tool(text: str):
    words = text.split()

    word_count = len(words)

    reading_time = round(word_count / 200, 2)

    keywords = list(set(words))[:5]
    
    summary = " ".join(words[:30])

    return {
        "word_count": word_count,
        "reading_time_minutes": reading_time,
        "keywords": keywords,
        "summary": summary
    }

### Data Transformation Tool

In [ ]:
from datetime import datetime
import re

def data_transformation_tool(operation: str, value: str):
    try:
        if operation == "km_to_miles":
            km = float(value)
            return {"result": round(km * 0.621371, 4)}

        elif operation == "c_to_f":
            c = float(value)
            return {"result": round((c * 9/5) + 32, 2)}

        elif operation == "format_date":
            date = datetime.strptime(value, "%Y-%m-%d")
            return {"result": date.strftime("%d %B %Y")}

        elif operation == "validate_email":
            pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
            return {"valid": bool(re.match(pattern, value))}

        else:
            return {"error": "Unknown operation"}

    except Exception as e:
        return {"error": str(e)}

## Build Tool Router

In [ ]:
def execute_tool(decision: AgentDecision):
    
    if decision.intent == "calculator":
        return calculator_tool(decision.expression)

    elif decision.intent == "text_processing":
        return text_processing_tool(decision.text)

    elif decision.intent == "data_transformation":
        return data_transformation_tool(
            decision.operation,
            decision.value
        )

    else:
        return {"error": "Unknown intent"}

## Final Response Formatting Layer

In [ ]:
def format_final_response(user_input: str, tool_output: dict):

    final_prompt = f"""
You are a professional system.

Generate a direct response using ONLY the tool result.
Do not add titles.
Do not add labels.
Do not add extra commentary.
Do not reinterpret the result.
Keep it concise.

User request:
{user_input}

Tool result:
{tool_output}
"""
    return llm.invoke(final_prompt)

## Full Agent Function

In [ ]:
def run_agent(user_input: str):

    # Step 1: LLM decision
    decision = get_agent_decision(user_input)

    # Step 2: Execute tool
    tool_output = execute_tool(decision)

    # Step 3: Format final response
    final_answer = format_final_response(user_input, tool_output)

    return final_answer

In [ ]:
print(run_agent("Convert 10 km to miles"))

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The conversion of 10 kilometers to miles is approximately 6.2137 miles.


In [ ]:
run_agent("Calculate (25 + 5) * 3")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'The calculation for (25 + 5) * 3 is equal to 90. Therefore, the final answer is 90.'

In [ ]:
run_agent("Analyze this text: Artificial intelligence is transforming industries worldwide by automating complex tasks and improving decision making.")

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Word count: 14\nReading time: 0.07 minutes\nKeywords: is, transforming, and, complex, industries\nSummary: Artificial intelligence is transforming industries worldwide by automating complex tasks and improving decision making.'